# res3 一键复现 Notebook

这个 notebook 设计成可以直接 `Run All`：
- 默认执行 **快速复现**：基于现有指标重建最终表格，最快、最稳。
- 可选执行 **重训练**：机器学习可以直接跑；深度学习需要 GPU 和完整资源。

如果你只想拿到最终论文表格，直接 Run All 就够了。
- 默认输出结果与 `res3` 目录保持一致。


In [ ]:
from pathlib import Path
import os
import sys
import json
import platform
import subprocess
from pprint import pprint
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists() and (ROOT / "res3_repro" / "scripts").exists():
    ROOT = (ROOT / "res3_repro").resolve()
elif not (ROOT / "scripts").exists() and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent.resolve()

os.chdir(ROOT)
print("ROOT:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


def run(cmd):
    print("$", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], check=True, cwd=ROOT)


In [ ]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    run([sys.executable, "-m", "pip", "install", "-U", "pip"])
    run([sys.executable, "-m", "pip", "install", "numpy<2", "-r", "requirements.txt"])
else:
    print("Skip dependency installation. If this is a new environment, set INSTALL_DEPS=True and rerun.")


In [ ]:
versions = {}
for name in ["numpy", "pandas", "sklearn", "torch", "transformers", "tqdm", "joblib"]:
    try:
        module = __import__(name)
        versions[name] = getattr(module, "__version__", "unknown")
    except Exception as exc:
        versions[name] = f"IMPORT FAILED: {exc}"

pprint(versions)

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        for index in range(torch.cuda.device_count()):
            print(index, torch.cuda.get_device_name(index))
except Exception as exc:
    print("Torch check failed:", exc)


## 运行开关

默认配置适合直接 `Run All`：
- `RUN_REBUILD_TABLE = True`：重建最终主表。
- 其他训练开关默认关闭，避免第一次运行就进入超长训练。

如果你要重训练，把对应开关改成 `True` 再从头运行。


In [ ]:
RUN_PREPARE_DATA = False
RUN_ML = False
RUN_DEEP_FC = False
RUN_DEEP_RCNN = False
RUN_REBUILD_TABLE = True
SHOW_TABLES = True

DATA_DIR = ROOT / "data" / "processed_110"
RAW_DIR = ROOT / "data" / "2018数据集"
PRETRAINED_ENCODER = ROOT / "outputs" / "recovered_bert_from_tuned_fc"

OUTPUT_ML = ROOT / "outputs_cmp_110_seed66" / "ml_baselines"
OUTPUT_FC = ROOT / "outputs_cmp_110_seed66" / "deep_hierarchical_fc_c2r_tune1"
OUTPUT_RCNN = ROOT / "outputs_cmp_110_seed66" / "deep_hierarchical_rcnn_c2r"
FINAL_DIR = ROOT / "outputs_cmp_110_res3opt"

paths_to_check = {
    "DATA_DIR": DATA_DIR,
    "RAW_DIR": RAW_DIR,
    "PRETRAINED_ENCODER": PRETRAINED_ENCODER,
    "OUTPUT_ML": OUTPUT_ML,
    "OUTPUT_FC": OUTPUT_FC,
    "OUTPUT_RCNN": OUTPUT_RCNN,
    "FINAL_DIR": FINAL_DIR,
}

for key, value in paths_to_check.items():
    print(f"{key}: {value} -> {value.exists()}")


In [ ]:
if RUN_PREPARE_DATA:
    if not RAW_DIR.exists():
        raise FileNotFoundError(f"Raw data directory not found: {RAW_DIR}")
    run([
        sys.executable,
        "scripts/prepare_data.py",
        "--data-dir", str(RAW_DIR),
        "--output-dir", str(DATA_DIR),
        "--target-size", "50000",
        "--top-k-labels", "110",
    ])
else:
    print("Skip data preparation.")


In [ ]:
if RUN_ML:
    run([
        sys.executable,
        "scripts/train_ml_baselines.py",
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(OUTPUT_ML),
        "--models", "svm", "sgd", "pa",
        "--seed", "66",
    ])
else:
    print("Skip ML training.")


In [ ]:
if RUN_DEEP_FC:
    if not PRETRAINED_ENCODER.exists():
        raise FileNotFoundError(f"Pretrained encoder not found: {PRETRAINED_ENCODER}")
    run([
        sys.executable,
        "scripts/train_deep_hierarchical.py",
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(OUTPUT_FC),
        "--fine-model-type", "fc",
        "--coarse-model-type", "rcnn",
        "--pretrained-model", str(PRETRAINED_ENCODER),
        "--max-length", "384",
        "--train-batch-size", "12",
        "--eval-batch-size", "32",
        "--learning-rate", "1.2e-5",
        "--dropout", "0.1",
        "--epochs", "5",
        "--early-stopping-patience", "2",
        "--selection-metric", "accuracy",
        "--fusion-selection-metric", "accuracy",
        "--max-fusion-top-k-coarse", "3",
        "--seed", "66",
    ])
else:
    print("Skip deep FC hierarchical training.")


In [ ]:
if RUN_DEEP_RCNN:
    if not PRETRAINED_ENCODER.exists():
        raise FileNotFoundError(f"Pretrained encoder not found: {PRETRAINED_ENCODER}")
    run([
        sys.executable,
        "scripts/train_deep_hierarchical.py",
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(OUTPUT_RCNN),
        "--fine-model-type", "rcnn",
        "--coarse-model-type", "rcnn",
        "--pretrained-model", str(PRETRAINED_ENCODER),
        "--max-length", "256",
        "--train-batch-size", "16",
        "--eval-batch-size", "64",
        "--learning-rate", "8e-6",
        "--dropout", "0.1",
        "--epochs", "4",
        "--early-stopping-patience", "2",
        "--selection-metric", "accuracy",
        "--fusion-selection-metric", "accuracy",
        "--max-fusion-top-k-coarse", "2",
        "--seed", "66",
    ])
else:
    print("Skip deep RCNN hierarchical training.")


In [ ]:
if RUN_REBUILD_TABLE:
    run([sys.executable, "scripts/build_res3_table.py"])
else:
    print("Skip final table rebuild.")


In [ ]:
summary_path = FINAL_DIR / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Final summary:")
    pprint(summary)
else:
    print("Summary file not found:", summary_path)


In [ ]:
if SHOW_TABLES:
    try:
        import pandas as pd
        display(pd.read_csv(FINAL_DIR / "results_table.csv"))
        display(pd.read_csv(FINAL_DIR / "results_table_contrast.csv"))
    except Exception as exc:
        print("Pandas display failed:", exc)
        print("Check files manually:")
        print(FINAL_DIR / "results_table.csv")
        print(FINAL_DIR / "results_table_contrast.csv")
else:
    print("Skip table display.")


## 结果位置

执行完成后重点看这几个文件：
- `outputs_cmp_110_res3opt/results_table.csv`
- `outputs_cmp_110_res3opt/results_table_contrast.csv`
- `outputs_cmp_110_res3opt/requirements_check.md`
- `outputs_cmp_110_res3opt/summary.json`

如果你只是论文提交，默认 `Run All` 到这里就够了。
